# 🩺 DoCare AI - Fine-Tuning Llama 3.2 on Kaggle (2x T4 GPU)

Notebook ini dioptimalkan khusus untuk lingkungan **Kaggle GPU (2x Tesla T4 atau P100)**.

### Fitur Training:
1. **Model**: `meta-llama/Llama-3.2-3B-Instruct` atau `unsloth/Llama-3.2-3B-Instruct`
2. **Teknik**: QLoRA (Quantized Low-Rank Adaptation) 4-bit (NF4)
3. **Hardware Kaggle**: 2x T4 GPU (32 GB VRAM) + 30 GB RAM
4. **Dataset**: MedQuAD (Medical Question Answering Dataset)
5. **Output**: LoRA Adapter + Konversi ke GGUF untuk dijalankan di **Ollama lokal**!

In [ ]:
# 1. Pastikan Internet dan GPU Aktif di Kaggle Settings (Sidebar kanan -> Settings -> Internet ON, Accelerator GPU T4 x2)
!nvidia-smi

In [ ]:
# 2. Instalasi Library Terbaru (Unsloth / PEFT / TRL)
!pip install -q -U "transformers>=4.45.0" "datasets>=2.20.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" trl accelerate

In [ ]:
# 3. Import Library & Setup Multi-GPU
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

device_count = torch.cuda.device_count()
print(f"Jumlah GPU terdeteksi di Kaggle: {device_count}")
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 4. Muat Dataset MedQuAD (Otomatis dari GitHub DoCare AI / HuggingFace)
# Kita membaca langsung dataset MedQuAD yang bersih
url = "https://raw.githubusercontent.com/StevenHarun/DoCareAi/main/data/cleaned_medquad.csv"
try:
    df = pd.read_csv(url)
    print(f"Dataset berhasil dimuat dari GitHub! Total data: {len(df)}")
except Exception:
    # Fallback ke dataset MedQuAD publik di Hugging Face
    from datasets import load_dataset
    hf_data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    df = hf_data.to_pandas()
    df = df.rename(columns={"Questions": "question", "Answers": "answer"})
    print(f"Dataset dimuat dari HuggingFace MedQuAD! Total data: {len(df)}")

df = df.dropna(subset=["question", "answer"])
df_train = df.sample(n=min(5000, len(df)), random_state=42)
print(f"Jumlah sampel untuk fine-tuning: {len(df_train)}")

In [ ]:
# 5. Muat Base Model dengan 4-bit Quantization (Ringan & Cepat)
model_id = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# 6. Konfigurasi LoRA Adapter (Targeting seluruh Attention & MLP layers)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 7. Format Prompt Medis DoCare AI (Llama 3 Chat Template)
def format_chat_prompt(batch):
    formatted_texts = []
    for q, a in zip(batch["question"], batch["answer"]):
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            f"Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya. Jawab dengan empatik, sopan, dan berdasarkan informasi medis yang akurat.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n{q}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\nTerima kasih telah berkonsultasi dengan DoCare AI. "
            f"{a} Semoga informasi ini membantu dan lekas sembuh.<|eot_id|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = Dataset.from_pandas(df_train)
dataset = dataset.map(format_chat_prompt, batched=True)

In [ ]:
# 8. Eksekusi Training di Kaggle GPU
output_dir = "/kaggle/working/docare_llama_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=200,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=100,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("🚀 Memulai Fine-Tuning di Kaggle...")
trainer.train()

# Simpan LoRA Adapter Final
model.save_pretrained(f"{output_dir}/final")
tokenizer.save_pretrained(f"{output_dir}/final")
print(f"✅ Training Selesai! Model tersimpan di {output_dir}/final")

In [ ]:
# 9. Uji Coba Inferensi DoCare AI Hasil Fine-Tuning
test_query = "Apa saja gejala awal penyakit glaukoma pada mata?"
prompt = (
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
    f"Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya.<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n{test_query}<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.3)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
# 10. Kompres Model Output agar Mudah Diunduh ke Laptop
!tar -czvf /kaggle/working/docare_llama_lora.tar.gz -C /kaggle/working/docare_llama_lora/final .
print("File /kaggle/working/docare_llama_lora.tar.gz siap diunduh dari tab Output Kaggle!")